In [ ]:
import pandas as pd
import numpy as np
import torch
from torch.utils.data import TensorDataset, DataLoader
from transformers import BertTokenizer, BertForSequenceClassification
from sklearn.preprocessing import LabelEncoder
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report
from torch.optim import AdamW

In [ ]:
train_df = pd.read_csv('nlp_train.csv')
test_df  = pd.read_csv('nlp_test.csv')

print(f'Train: {len(train_df)} | Test: {len(test_df)}')
print(train_df.head(3))

In [ ]:
le = LabelEncoder()
train_df['label'] = le.fit_transform(train_df['risk'])
test_df['label']  = le.transform(test_df['risk'])

print('Classes:', le.classes_)

In [ ]:
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

def tokenize(texts, labels, tokenizer, max_length=64):
    encoded = tokenizer(
        texts,
        padding='max_length',
        truncation=True,
        max_length=max_length,
        return_tensors='pt'
    )
    return {
        'input_ids':      encoded['input_ids'],
        'attention_mask': encoded['attention_mask'],
        'labels':         torch.tensor(labels, dtype=torch.long)
    }

train_data = tokenize(train_df['text'].tolist(), train_df['label'].tolist(), tokenizer)
test_data  = tokenize(test_df['text'].tolist(),  test_df['label'].tolist(),  tokenizer)

In [ ]:
train_dataset = TensorDataset(
    train_data['input_ids'],
    train_data['attention_mask'],
    train_data['labels']
)

test_dataset = TensorDataset(
    test_data['input_ids'],
    test_data['attention_mask'],
    test_data['labels']
)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
test_loader  = DataLoader(test_dataset,  batch_size=16, shuffle=False)

print(f'Train batches: {len(train_loader)}')
print(f'Test batches : {len(test_loader)}')

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

classes = np.unique(train_df['label'])
weights = compute_class_weight('balanced', classes=classes, y=train_df['label'])
weights_tensor = torch.tensor(weights, dtype=torch.float).to(device)

model = BertForSequenceClassification.from_pretrained(
    'bert-base-uncased',
    num_labels=3
)
model = model.to(device)

criterion = torch.nn.CrossEntropyLoss(weight=weights_tensor)
optimizer = AdamW(model.parameters(), lr=3e-5)

In [ ]:
def train_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss, correct = 0, 0

    for input_ids, attention_mask, labels in loader:
        input_ids      = input_ids.to(device)
        attention_mask = attention_mask.to(device)
        labels         = labels.to(device)

        optimizer.zero_grad()
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        loss    = criterion(outputs.logits, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        correct    += (outputs.logits.argmax(dim=1) == labels).sum().item()

    acc = correct / len(loader.dataset)
    return total_loss / len(loader), acc


EPOCHS    = 30
best_acc  = 0
patience  = 3
no_improve = 0

for epoch in range(EPOCHS):
    loss, acc = train_epoch(model, train_loader, optimizer, criterion)
    print(f'Epoch {epoch+1}/{EPOCHS} — Loss: {loss:.4f} | Acc: {acc:.4f}')

    if acc > best_acc:
        best_acc   = acc
        no_improve = 0
        torch.save(model.state_dict(), 'best_model.pt')
        print(f'Best saved (acc: {best_acc:.4f})')
    else:
        no_improve += 1
        print(f'No improve {no_improve}/{patience}')
        if no_improve >= patience:
            print(f'Early stopping at epoch {epoch+1}')
            break

model.load_state_dict(torch.load('best_model.pt'))
print(f'Best Acc: {best_acc:.4f}')

In [ ]:
def evaluate(model, loader):
    model.eval()
    all_preds, all_labels = [], []

    with torch.no_grad():
        for input_ids, attention_mask, labels in loader:
            input_ids      = input_ids.to(device)
            attention_mask = attention_mask.to(device)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            preds   = outputs.logits.argmax(dim=1)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.numpy())

    print(classification_report(all_labels, all_preds, target_names=le.classes_))

evaluate(model, test_loader)

In [ ]:
def predict(text):
    model.eval()
    encoded = tokenizer(
        text,
        padding='max_length',
        truncation=True,
        max_length=64,
        return_tensors='pt'
    )
    input_ids      = encoded['input_ids'].to(device)
    attention_mask = encoded['attention_mask'].to(device)

    with torch.no_grad():
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        pred    = outputs.logits.argmax(dim=1).item()

    return le.inverse_transform([pred])[0]

text = "Dermatology case: 45 year old male patient. Lesion location: back. Diagnosis method: histo."
print('Risk:', predict(text))

In [ ]:
# import pickle

# model.save_pretrained('nlp_model')
# tokenizer.save_pretrained('nlp_model')

# with open('nlp_model/label_encoder.pkl', 'wb') as f:
#     pickle.dump(le, f)

# print('Saved!')

In [ ]:
# import shutil

# shutil.make_archive('nlp_model', 'zip', 'nlp_model')

# from google.colab import files
# files.download('nlp_model.zip')

In [ ]:
# import pickle

# model     = BertForSequenceClassification.from_pretrained('nlp_model')
# tokenizer = BertTokenizer.from_pretrained('nlp_model')

# with open('nlp_model/label_encoder.pkl', 'rb') as f:
#     le = pickle.load(f)

# model = model.to(device)
# print('Model loaded!')